# LLM 数学推理评估项目（GSM8K Best-of-N & ProcessBench）

本项目旨在评估小型大语言模型 **Qwen/Qwen2.5-0.5B-Instruct** 在 **GSM8K** 数学推理数据集上的表现。项目结合 **Best-of-N（BoN）采样策略** 与 **ProcessBench** 的细粒度过程监督，对不同 **过程奖励模型（Process Reward Model，PRM）** 及其 **聚合策略（Aggregation Strategy）** 进行系统评估，分析当前模型在数学推理任务中的优势与瓶颈。

---

# 项目概览

整个实验流程包括以下五个阶段：

1. 数据集基线测试（Dataset Baseline）
2. 候选推理轨迹生成（Generation）
3. Oracle 理论上限评估（Oracle Evaluation）
4. 奖励模型外部评估（Extrinsic Evaluation）
5. ProcessBench 内在评估（Intrinsic Evaluation）

---

# 环境配置

## 硬件环境

- NVIDIA RTX 4060（Windows 本地环境）

## 核心依赖

- Python 3.13
- PyTorch
- Transformers
- Accelerate

## 性能优化

实验过程中进行了若干优化，以提高推理效率：

- 强制模型张量使用 `.to("cuda")`，解决 GPU 利用率过低的问题。
- 修复 `transformers` 中 `torch_dtype` 参数的弃用警告。
- 优化 Best-of-N 批量推理流程，提高评测效率。

---

# 实验流程

## 1. 数据集基线测试（Dataset Baseline）

在进行自定义数据生成和奖励模型评估之前，首先对 **ProcessBench** 构建的 GSM8K 候选轨迹数据集进行了基线测试。

### 测试方式

- 每道题随机抽取一条推理轨迹（即 **BoN@1**）
- 不进行任何奖励模型排序

### 运行命令

```powershell
python eval_bon.py ^
    --eval_file data/processbench_bon_gsm8k.jsonl ^
    --head gru ^
    --checkpoint checkpoints/gru_clean.pt ^
    --ks 1
```

### 实验结果

| 指标 | 数值 |
|------|------|
| BoN@1 | **49.88%** |

> 共进行 **7500 次随机抽样**。

### 结论

在完全不依赖奖励模型排序的情况下，随机选择一条推理轨迹即可达到约 **49.88%** 的正确率。

这一结果可以视为数据集自身的**天然基线（Base Accuracy）**，也是后续评估 PRM 重排序能力的重要参考。

---

## 2. 候选推理轨迹生成（Generation）

为了评估 Best-of-N 的潜力，使用生成模型重新生成 GSM8K 测试集的候选推理轨迹。

### 生成模型

```
Qwen/Qwen2.5-0.5B-Instruct
```

### 参数设置

| 参数 | 数值 |
|------|------|
| 数据集 | GSM8K Test |
| 题目数量 | 1319 |
| 每题采样数 | 16 |
| Batch Size | 16 |

### 输出文件

```text
data/gsm8k_qwen0.5b_bon16.jsonl
```

### 状态

> ✅ 已完成

---

## 3. Oracle 理论上限评估（Oracle Evaluation）

在接入奖励模型之前，首先计算生成器的理论性能上限，即：

> **16 条候选轨迹中是否至少存在一条正确答案。**

### 实验结果

| 指标 | 数值 |
|------|------|
| Oracle@16 | **76.04%** |

### 结论

对于 GSM8K，大约 **76%** 的题目能够在 16 次采样中至少生成一个正确答案。

说明：

- 当前生成模型已经具备较强的采样能力；
- 系统整体性能瓶颈主要来自**奖励模型（Reward Model）**，而非生成模型。

---

## 4. 奖励模型外部评估（Extrinsic Evaluation）

使用 **GRU 过程奖励模型（PRM）** 对生成的 16 条候选轨迹进行评分，并根据不同聚合策略完成 Best-of-N 重排序。

### 奖励模型

```text
checkpoints/gru_clean.pt
```

同时修改了 `eval_bon.py` 中的聚合逻辑，对不同 Aggregation Strategy 进行了消融实验。

### 实验结果

| 聚合策略 | 说明 | BoN@16 |
|-----------|------|--------|
| **last** | 仅使用最后一步得分（ORM 模式） | **34.04%** 🏆 |
| **min** | 所有步骤取最低分（严格 PRM） | 32.37% |
| **sum** | 所有步骤得分求和 | 32.22% |
| **mean** | 所有步骤得分平均 | 31.99% |

---

## 5. ProcessBench 内在评估（Intrinsic Evaluation）

为了探究 GRU 外部评估表现不佳的根本原因，基于 **processbench_bon_gsm8k.jsonl** 数据集，开发了 `eval_intrinsic.py` 脚本，直接将奖励模型作为二分类器，测试其对完整解题轨迹正误的判别能力。

> **说明：** 当前 ProcessBench 数据集版本仅提供整条候选轨迹的布尔标签（`label: true/false`），因此本实验评估对象为**轨迹级（Trajectory-level）**，并采用表现最优的 **last** 聚合策略作为最终预测分数。

### 实验结果

| 指标 | 数值 | 说明 |
|------|------|------|
| ROC-AUC | **0.6139** | 排序区分能力，仅略高于随机猜测（0.50） |
| Accuracy | **0.5000** | 在 0.5 阈值下的二分类准确率，接近随机水平 |

> 实验基于 **400 条独立验证轨迹**完成。

### 状态

> ✅ 已完成

---

# 实验结论

## 1. 生成端能力充足，判别端成为系统瓶颈

实验结果表明，当前系统的主要瓶颈并非生成模型，而是奖励模型。

- Oracle@16 达到 **76.04%**
- GRU 最优 BoN@16 仅 **34.04%**
- Intrinsic Evaluation 中 ROC-AUC 仅 **0.6139**

说明生成模型能够较高概率产生正确答案，但奖励模型缺乏有效识别正确轨迹的能力，因此无法充分发挥 Best-of-N 采样优势。

---

## 2. GRU 的长程依赖建模能力不足

所有聚合策略中，**last**（仅使用最后一步得分）取得了最佳效果，而理论上更符合 PRM 思想的 **min**、**mean** 等策略反而表现更差。

结合较低的 ROC-AUC，可以推断：

- GRU 在处理长推理链时存在明显的长期记忆遗忘（Long-range Forgetting）；
- 随着推理长度增加，早期步骤的信息逐渐被后续状态覆盖；
- 难以建立跨步骤（Cross-step Dependency）的全局逻辑关联。

因此，GRU 更接近于结果奖励模型（Outcome Reward Model, ORM），尚未充分体现过程奖励模型（PRM）的优势。

---

## 3. 奖励模型尚未有效学习逻辑正确性

在轨迹级分类任务中：

- Accuracy 仅为 **50%**；
- ROC-AUC 仅为 **0.6139**。

这表明模型输出分数的判别能力较弱，无法形成清晰的正确与错误轨迹分界，也难以承担数学推理中的有效重排序任务。

---

# 后续工作

基于当前实验结果，后续研究将重点围绕奖励模型展开优化：

- **架构升级**：弃用 GRU 奖励头，尝试基于 Transformer（Attention）的奖励模型，提高长程依赖建模能力。
- **重新验证**：完成新模型训练后，使用 `eval_intrinsic.py` 重新评估 ROC-AUC，目标显著超过当前 **0.6139**。
- **聚合策略研究**：在新的奖励模型基础上重新测试 `min`、`mean` 等过程监督聚合策略，验证 PRM 的实际收益。
- **扩展生成模型**：进一步测试更大规模生成模型（如 Qwen2.5-1.5B、Qwen2.5-3B），探索生成能力与奖励模型之间的协同效果。

### 项目总结：基于轻量级过程奖励模型 (PRM) 的数学推理评估

本项目旨在通过构建和评估轻量级的过程奖励模型（PRM），来优化数学推理轨迹的打分机制。我们利用冻结的语义编码器进行特征预计算，在此基础上快速训练并对比了多种轻量级神经网络架构（MLP、CNN、GRU）在成对 Q 值匹配（PQM）损失函数下的表现。

#### 核心项目流程
*   **阶段一：特征预计算 (Stage 1)** —— 利用冻结的 Qwen2.5-0.5B 模型，对全量数学数据进行前向传播，提取并缓存隐藏层特征。
*   **阶段二：轻量级网络头构建 (Stage 2)** —— 设计并实例化了小参数量、高效率的对比网络架构，用于接收缓存的特征。
*   **阶段三：对比训练与评估 (Stage 3)** —— 在轨迹级别进行优化训练，并对模型的打分排序能力进行多维度测试。

---

#### 1. 数据准备与防泄漏策略
为了确保模型评估的绝对严谨，我们将庞大的 Math-Shepherd 数据集严格按 99% 与 1% 的比例，切分为了互不重合的训练集与测试集，从根源上杜绝了数据泄露。

在处理这四十多万条数据的切分时，我们遭遇并克服了 Windows 系统经典的 I/O 缓冲问题。由于数据量极其庞大，常规的简短指令会导致数据滞留在内存中无法完全写入硬盘。我们通过引入强制落盘和清理缓冲区的机制，确保了大规模数据集的物理级安全写入。

#### 2. 奖励网络训练与调优
我们基于纯净的特征缓存，让三种不同的网络架构在完全相同的起跑线上进行了训练。

在初期，模型遭遇了深度学习中常见的梯度爆炸问题（Loss 变为 NaN）。经过排查，我们将初始学习率下调了十倍。这一关键超参数的修复瞬间稳住了训练阵脚，三种模型随后均在 3 个 Epoch 内呈现出健康的 Loss 下降趋势并成功收敛。

#### 3. 单步轨迹评估 (期中测验)
在完成训练后，我们对模型进行了成对分离度（Pairwise Separation）的测试，即评估模型将“正确步骤”的打分排在“错误步骤”前面的概率。在这个过程中，我们排除了两个工程隐患：
*   **兼容性修复**：统一了不同数据集格式下的标签键名差异。
*   **维度冲突修复**：由于数学题的解答步数长短不一，张量拼接时会发生崩溃。我们通过引入掩码逻辑，精准提取了每道题“最后一个有效步骤”的标签，成功将步骤级数据转化为整题级数据，打通了评估链路。

**评估结果对比**

| 网络架构 | 参数量级 | 成对分离度 (Pairwise Separation) | 架构特点与结论 |
| :--- | :--- | :--- | :--- |
| **MLP** | 约 26 万 | **0.6905** | 作为基线前馈网络，表现达标。 |
| **CNN** | 约 39 万 | **0.7067** | 能够捕捉局部相邻步骤的关联，分数获得显著提升。 |
| **GRU** | 约 79 万 | **0.7112** | **全场最佳。** 证明了具有时序记忆功能的循环网络，天生最适合处理数学推导这种高度依赖前后文的串行逻辑任务。 |

> **学术洞察：**
> 虽然 GRU 相比 MLP 在绝对准确率上仅提升了约 2%，但这代表它在随机基线（50%）之上，多挖掘出了超过 10% 的有效特征。在动辄十几步的复杂数学推理中，这种对序列依赖的精准把控，能极大地降低误差的指数级累积。

#### 4. Best-of-N (BoN) 终极评估 (待推进)
为了对齐 PRM 领域的最高学术标准，我们正着手推进 Best-of-N 评估，即测试模型能否从同一道题的 N 种不同解答路径中，精准挑出唯一正确的答案。

目前我们已经修复了 Windows 系统默认编码导致的文本读取乱码问题。当前面临的主要阻点在于数据结构不匹配：BoN 评估要求数据集必须是“按题分组”的层级化结构（包含候选解答列表），而我们目前使用的是随机切分的扁平化数据。接下来的计划是寻找匹配的专属 BoN 测试集，或者通过重构脚本，将现有测试集强行按题目进行聚类转换，以完成最终的测试。

In [1]:
from datasets import load_dataset
import random
import json

# 抽取数量（100~200之间任选）
N = 150

# 固定随机种子，保证结果可复现
SEED = 42

# 加载训练集
dataset = load_dataset(
    "peiyi9979/Math-Shepherd",
    split="train"
)

# 随机抽样
random.seed(SEED)
indices = random.sample(range(len(dataset)), N)

# 保存为 JSONL
with open("dummy_train.jsonl", "w", encoding="utf-8") as f:
    for idx in indices:
        json.dump(dataset[idx], f, ensure_ascii=False)
        f.write("\n")

print(f"Saved {N} samples to dummy_train.jsonl")

f:\anaconda3\envs\node2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved 150 samples to dummy_train.jsonl


In [2]:
from datasets import load_dataset
import json

# 加载 Math-Shepherd 数据集（请替换为具体的 HF 仓库名，如 "math-shepherd/..."）
dataset = load_dataset(
    "peiyi9979/Math-Shepherd",
    split="train"
)

# 导出为你代码需要的 jsonl 格式
output_file = "data/math_shepherd_full.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        
print(f"下载完成！共 {len(dataset)} 条数据，已保存至 {output_file}")

KeyboardInterrupt: 